# 10M-token FineWeb-Edu: Transformer vs original PPLSTM CUDA Graphs vs fused PPLSTM CUDA Graphs

**Purpose: establish whether the T4 CUDA fusion changes the PPLSTM mathematics *or training*, while comparing against a parameter-matched Transformer.**

Three runs (10,000,000 training-token presentations **per model**, not total):

1. **Transformer (SDPA + RoPE)**: four causal attention blocks, parameter-matched to the hybrid; fused SDPA requested when the GPU supports it.
2. **Hybrid original**: official xLSTM **mLSTM → PPLSTM → mLSTM → PPLSTM**, using your unmodified published PyTorch PPLSTM and CUDA Graph forward/backward.
3. **Hybrid fused T4**: **identical model, initialization, parameter shapes, batches and optimizer** to #2. Its only difference is that two PPLSTM memory scans, normalizers and output calculations use the experimental T4 CUDA kernels. Its permutation routing, signed gates and all other layers remain the original.

**Why this is a better maths test:** the two hybrids begin from identical weights. The notebook checks the original and CUDA-fused *same-weight* outputs and parameter gradients, and directly checks PPLSTM input gradients. It **repeats the full-model same-weight checks using trained weights every ~2M tokens**. It separately logs the divergence of their independently trained weights and held-out losses.

**GPU**: Use Kaggle **Settings → Accelerator → GPU T4 x1**, attach your previously used `cartman` FineWeb-Edu sample-10BT parquet dataset (or a compatible prepared tokenizer and token-split folder), and enable Internet for `xlstm`, `cupy-cuda12x`, `sentencepiece`, `pyarrow` if needed. This is a separate notebook and **does not interact with your running 200M job**.

FP32 for both hybrids. CUDA Graphs capture forward/backward only; optimizer steps and gradient clipping run outside. The Transformer uses optional FP16 only inside attention, as in the 200M experiment. The numerical comparisons are **hybrid vs hybrid**, not Transformer vs hybrid (different architectures). A failing correctness check stops training. The ZIP contains only small CSV/JSON reports and CUDA source; `.pt` checkpoints are excluded.

**Important**: tiny floating-point differences can accumulate over training. Similar final loss alone does **not** establish equivalence, and bitwise-identical results are not promised. Accuracy/speed claims apply only to the tested T4 configurations; real GPU execution has not been verified for this new combined notebook.

**Dataset-loading fix v2 (16 Nov 2026):** corrected an undefined `write_json` call to the already-defined `save_json`. Mathematical models, CUDA kernels, learning setup, and validation checks unchanged.


**v3 reliability fix:** The strict same-weights audit no longer passes its `passed` flag twice to `check()`. No memory equations, CUDA code, data, model settings or training budget changed.


## 1 · Settings, dependencies and GPU

In [ ]:
# --- Edit settings here ---
TARGET_TOKENS_PER_MODEL = 10_000_000
WIDTH, DEPTH, HEADS = 72, 4, 4
CONTEXT, BATCH = 128, 8
SEED = 20261116
REPORT_EVERY_STEPS = 500
AUDIT_EVERY_TOKENS = 2_000_000
MAX_TOTAL_HOURS = 2.5
RUN_SMOKE_ONLY = False                # True: only ~32 optimizer steps/model (check notebook quickly)
SMOKE_STEPS = 32
TRAIN_PARQUET_GLOB = '/kaggle/input/**/000_00000.parquet'  # cartman / FineWeb-Edu sample-10BT
PREPARED_DIR = ''                    # optional folder containing train.bin valid.bin test.bin spm.model
MAX_PREP_TRAIN_TOKENS = 12_000_000
SMALL_VALIDATION_TOKENS = 8_192
FINAL_EVAL_TOKENS = 32_768
USE_FP16_SDPA_ATTENTION = True       # Transformer only; hybrid comparisons always FP32
REQUIRE_CUDA_GRAPHS_FOR_HYBRIDS = True
SAVE_FINAL_CHECKPOINTS = True       # checkpoints remain separate from the ZIP

import os,sys,subprocess,importlib.util,time,random,math,copy,glob,json,hashlib,zipfile,traceback,statistics
from pathlib import Path
OUT=Path('/kaggle/working/pplstm_fusion_10m' if Path('/kaggle/working').exists() else '/mnt/data/pplstm_fusion_10m')
OUT.mkdir(parents=True,exist_ok=True)
need=[]
for module,pkg in [('xlstm','xlstm==2.0.6'),('cupy','cupy-cuda12x'),('sentencepiece','sentencepiece'),('pyarrow','pyarrow')]:
    if importlib.util.find_spec(module) is None:need.append(pkg)
if need:
    print('Installing:',need,flush=True)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import numpy as np,pandas as pd,torch,cupy as cp
from torch import nn
from torch.nn import functional as F
assert torch.cuda.is_available(),'Enable a Kaggle GPU in Notebook Settings before running.'
DEVICE=torch.device('cuda:0')
CAPABILITY=torch.cuda.get_device_capability(0)
print('GPU:',torch.cuda.get_device_name(0),'capability:',CAPABILITY,'Torch:',torch.__version__,'CUDA:',torch.version.cuda,'CuPy:',cp.__version__)
if CAPABILITY!=(7,5):
    print('WARNING: This CUDA kernel was verified on a T4 (7.5); this GPU differs. Treat as an exploratory port.')
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
RUN_START=time.monotonic();DEADLINE=RUN_START+MAX_TOTAL_HOURS*3600
CHECKS=[];CHECK_FAILURES=[]

def save_json(name,obj):
    path=OUT/name
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,default=str))
    os.replace(tmp,path)

def save_csv(name,rows):
    if rows:pd.DataFrame(rows).to_csv(OUT/name,index=False)

def nparams(model):return sum(p.numel() for p in model.parameters() if p.requires_grad)

def bundle():
    output=OUT/'PPLSTM_Transformer_Fusion_10M_Results.zip'
    with zipfile.ZipFile(output.with_suffix('.partial.zip'),'w',zipfile.ZIP_DEFLATED) as z:
        for path in [*OUT.glob('*.csv'),*OUT.glob('*.json'),*OUT.glob('*.cu')]:z.write(path,path.name)
    os.replace(output.with_suffix('.partial.zip'),output)
    return output

def check(name,passed,**data):
    row={'name':name,'passed':bool(passed),**data};CHECKS.append(row)
    save_json('correctness_checks.json',CHECKS)
    print(('PASS' if passed else 'FAIL'),name,{k:v for k,v in data.items() if k!='per_parameter'},flush=True)
    if not passed:CHECK_FAILURES.append(name)
    return bool(passed)

def skip(name,error):
    print('ERROR',name,repr(error),flush=True)
    CHECK_FAILURES.append(name)
    save_json('error_'+name.replace('/','_')+'.json',{'error':repr(error),'traceback':traceback.format_exc()})

## 2 · Published PPLSTM source: same gates, permutations, initialization and scans

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Adapted from the original PPLSTM project by Rickey (https://github.com/rickey1990/PPLSTM/blob/main/pplstm.py)
# The transition equations and scan implementation are kept as published.
"""PPLSTM: associative 3-channel permutation-mixed recurrent cell."""
import itertools
import math
import torch
from torch import Tensor, nn
from torch.nn import functional as F


def shift_right(x: Tensor) -> Tensor:
    return torch.cat((torch.zeros_like(x[:, :1]), x[:, :-1]), dim=1)


def scalar_affine_scan(a: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    A, B = a, b
    step = 1
    while step < a.shape[1]:
        A, B = (
            torch.cat((A[:, :step], A[:, step:] * A[:, :-step]), dim=1),
            torch.cat((B[:, :step], B[:, step:] + A[:, step:] * B[:, :-step]), dim=1),
        )
        step *= 2
    return A, B


def matrix_affine_scan(A: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    step = 1
    while step < A.shape[1]:
        next_A = torch.cat((A[:, :step], A[:, step:] @ A[:, :-step]), dim=1)
        next_b = torch.cat((
            b[:, :step],
            b[:, step:] + (A[:, step:] @ b[:, :-step].unsqueeze(-1)).squeeze(-1),
        ), dim=1)
        A, b = next_A, next_b
        step *= 2
    return A, b


class PPLSTM(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        if width < 3 or width % 3 != 0:
            raise ValueError('width must be a positive multiple of 3')
        self.width = width
        self.groups = width // 3
        self.xproj = nn.Linear(width, 4 * width)
        self.mix_logit = nn.Parameter(torch.full((4, width), math.log(3.0)))
        self.sign_proj = nn.Linear(width, width)
        self.initial_cell = nn.Parameter(torch.linspace(-1.5, 1.5, 3).repeat(self.groups))
        self.input_bias = nn.Parameter(torch.full((width,), -4.0))
        self.forget_bias = nn.Parameter(torch.full((width,), 4.0))
        nn.init.zeros_(self.sign_proj.weight)
        nn.init.constant_(self.sign_proj.bias, 2.8)

        bank = []
        for order in itertools.permutations(range(3)):
            matrix = torch.zeros(3, 3)
            for row, column in enumerate(order):
                matrix[row, column] = 1.0
            bank.append(matrix)
        self.register_buffer('permutation_bank', torch.stack(bank))

        self.selector = nn.Linear(width, self.groups * 6)
        nn.init.zeros_(self.selector.weight)
        nn.init.constant_(self.selector.bias, -1.0)
        with torch.no_grad():
            for group in range(self.groups):
                self.selector.bias[group * 6] = 3.0

    def _token_maps(self, x: Tensor):
        if x.ndim != 3 or x.shape[-1] != self.width:
            raise ValueError('Expected x with shape [batch, time, width]')
        batch, length, width = x.shape
        group_count = self.groups
        previous = shift_right(x)
        coeff = self.mix_logit.sigmoid().view(1, 1, 4, width)
        gates = (self.xproj(x).reshape(batch, length, 4, width) * coeff
                 + self.xproj(previous).reshape(batch, length, 4, width) * (1 - coeff))
        input_logits, forget_logits, candidate_logits, output_logits = gates.unbind(2)
        mixed = x * self.mix_logit[0].sigmoid() + previous * (1 - self.mix_logit[0].sigmoid())
        signs = self.sign_proj(mixed).tanh().reshape(batch, length, group_count, 3)
        z = candidate_logits.tanh().reshape(batch, length, group_count, 3)
        log_i = (input_logits + self.input_bias).reshape(batch, length, group_count, 3).mean(-1)
        log_f = F.logsigmoid((forget_logits + self.forget_bias)
                           .reshape(batch, length, group_count, 3).mean(-1))
        f_prefix = log_f.cumsum(dim=1)
        m = f_prefix + torch.cummax(torch.cat((
            torch.zeros_like(f_prefix[:, :1]), log_i - f_prefix,
        ), dim=1), dim=1).values[:, 1:]
        f = (log_f + shift_right(m) - m).exp()
        i = (log_i - m).exp()
        weights = self.selector(mixed).reshape(batch, length, group_count, 6).softmax(-1)
        P = torch.einsum('btgp,pij->btgij', weights, self.permutation_bank)
        A = f[..., None, None] * P * signs[..., None, :]
        b = i[..., None] * z
        return A, b, f, i, output_logits

    def forward(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        prefix_A, prefix_b = matrix_affine_scan(A, b)
        c0 = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        c = (prefix_A @ c0[:, None, :, :, None]).squeeze(-1) + prefix_b
        norm_A, norm_b = scalar_affine_scan(f, i)
        n = norm_A + norm_b
        ratio = (c / n[..., None].clamp_min(1e-12)).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

    def serial_reference(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        c = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        n = x.new_ones(batch, self.groups)
        states = []
        for t in range(length):
            c = (A[:, t] @ c.unsqueeze(-1)).squeeze(-1) + b[:, t]
            n = f[:, t] * n + i[:, t]
            states.append(c / n[..., None].clamp_min(1e-12))
        ratio = torch.stack(states, dim=1).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

## 3 · Experimental T4 fusion: original token maps; fused 3×3 scan, normalizer, output and reverse-gradient scan

In [ ]:
CUDA_SOURCE = '\n// NVRTC: no host C math.h header needed; expf/fmaxf are CUDA device built-ins.\n\n// One CTA per (batch, group), one thread per token. Elements beyond T are identity maps.\n// Memory: A_t (3x3), b_t (3), f_t, i_t. Fused scan output is normalized and gated.\nextern "C" __global__ void pplstm_fused_fwd(\n const float *__restrict__ A, const float *__restrict__ b,\n const float *__restrict__ f, const float *__restrict__ i,\n const float *__restrict__ o, const float *__restrict__ c0,\n float *__restrict__ h, float *__restrict__ cstate, float *__restrict__ nstate,\n int T, int G) {\n  const int lane=threadIdx.x;\n  const int group=blockIdx.x % G;\n  const int batch=blockIdx.x / G;\n  const int gidx=(batch*T+lane)*G+group;\n  const int bidx=gidx*3, aidx=gidx*9;\n  float aa[9], bb[3];\n  float ff=1.f, ii=0.f;\n  #pragma unroll\n  for(int j=0;j<9;++j) aa[j]=(j%4==0)?1.f:0.f;\n  bb[0]=0.f;bb[1]=0.f;bb[2]=0.f;\n  if(lane<T) {\n    #pragma unroll\n    for(int j=0;j<9;++j) aa[j]=A[aidx+j];\n    #pragma unroll\n    for(int j=0;j<3;++j) bb[j]=b[bidx+j];\n    ff=f[gidx];ii=i[gidx];\n  }\n  extern __shared__ float sh[];\n  for(int step=1;step<blockDim.x;step<<=1) {\n    float *own=sh+lane*14;\n    #pragma unroll\n    for(int j=0;j<9;++j) own[j]=aa[j];\n    #pragma unroll\n    for(int j=0;j<3;++j) own[9+j]=bb[j];\n    own[12]=ff;own[13]=ii;\n    __syncthreads();\n    if(lane>=step) {\n      const float *prev=sh+(lane-step)*14;\n      float na[9],nb[3];\n      #pragma unroll\n      for(int row=0;row<3;++row) {\n        #pragma unroll\n        for(int col=0;col<3;++col) {\n          na[row*3+col]=aa[row*3]*prev[col]\n                         +aa[row*3+1]*prev[3+col]\n                         +aa[row*3+2]*prev[6+col];\n        }\n        nb[row]=bb[row]+aa[row*3]*prev[9]\n                           +aa[row*3+1]*prev[10]\n                           +aa[row*3+2]*prev[11];\n      }\n      float next_ii=ii+ff*prev[13];\n      ff=ff*prev[12];ii=next_ii;\n      #pragma unroll\n      for(int j=0;j<9;++j) aa[j]=na[j];\n      #pragma unroll\n      for(int j=0;j<3;++j) bb[j]=nb[j];\n    }\n    __syncthreads();\n  }\n  if(lane<T) {\n    const int out=(batch*T+lane)*(G*3)+group*3;\n    const int start=(batch*G+group)*3;\n    float nn=fmaxf(ff+ii,1.e-12f);\n    nstate[gidx]=ff+ii;\n    #pragma unroll\n    for(int j=0;j<3;++j) {\n      float cc=bb[j]+aa[j*3]*c0[start]\n                     +aa[j*3+1]*c0[start+1]\n                     +aa[j*3+2]*c0[start+2];\n      cstate[bidx+j]=cc;\n      float sig=1.f/(1.f+expf(-o[out+j]));\n      h[out+j]=sig*(cc/nn);\n    }\n  }\n}\n\n// Reverse affine scan for the state and normalizer adjoints.\n// q_t = dL/dc_t + A_(t+1)^T q_(t+1),\n// u_t = dL/dn_t + f_(t+1) u_(t+1).\nextern "C" __global__ void pplstm_fused_bwd(\n const float *__restrict__ A, const float *__restrict__ f,\n const float *__restrict__ o, const float *__restrict__ c0,\n const float *__restrict__ cstate, const float *__restrict__ nstate,\n const float *__restrict__ gh,\n float *__restrict__ ga, float *__restrict__ gb,\n float *__restrict__ gf, float *__restrict__ gi,\n float *__restrict__ go, float *__restrict__ gc0,\n int T, int G) {\n  const int lane=threadIdx.x;\n  const int t=T-1-lane;\n  const int group=blockIdx.x%G, batch=blockIdx.x/G;\n  float aa[9],bb[3],ff=1.f,ii=0.f;\n  #pragma unroll\n  for(int j=0;j<9;++j) aa[j]=(j%4==0)?1.f:0.f;\n  bb[0]=0.f;bb[1]=0.f;bb[2]=0.f;\n  if(t>=0) {\n    int gidx=(batch*T+t)*G+group;\n    int bidx=gidx*3;\n    int out=(batch*T+t)*(G*3)+group*3;\n    float nn=nstate[gidx];\n    float denom=fmaxf(nn,1.e-12f);\n    float dn=0.f;\n    #pragma unroll\n    for(int j=0;j<3;++j) {\n      float sig=1.f/(1.f+expf(-o[out+j]));\n      float grad=gh[out+j];\n      float cc=cstate[bidx+j];\n      bb[j]=grad*sig/denom;\n      dn-=grad*sig*cc/(denom*denom);\n      go[out+j]=grad*(cc/denom)*sig*(1.f-sig);\n    }\n    ii=(nn>1.e-12f)?dn:0.f;\n    if(t+1<T) {\n      ff=f[(batch*T+t+1)*G+group];\n      int nextA=((batch*T+t+1)*G+group)*9;\n      #pragma unroll\n      for(int row=0;row<3;++row)\n        #pragma unroll\n        for(int col=0;col<3;++col)\n          aa[row*3+col]=A[nextA+col*3+row]; // transpose\n    }\n  }\n  extern __shared__ float sh[];\n  for(int step=1;step<blockDim.x;step<<=1) {\n    float *own=sh+lane*14;\n    #pragma unroll\n    for(int j=0;j<9;++j) own[j]=aa[j];\n    #pragma unroll\n    for(int j=0;j<3;++j) own[9+j]=bb[j];\n    own[12]=ff;own[13]=ii;\n    __syncthreads();\n    if(lane>=step) {\n      const float *prev=sh+(lane-step)*14;\n      float na[9],nb[3];\n      #pragma unroll\n      for(int row=0;row<3;++row) {\n        #pragma unroll\n        for(int col=0;col<3;++col)\n          na[row*3+col]=aa[row*3]*prev[col]\n                         +aa[row*3+1]*prev[3+col]\n                         +aa[row*3+2]*prev[6+col];\n        nb[row]=bb[row]+aa[row*3]*prev[9]\n                           +aa[row*3+1]*prev[10]\n                           +aa[row*3+2]*prev[11];\n      }\n      float new_ii=ii+ff*prev[13];\n      ff=ff*prev[12];ii=new_ii;\n      #pragma unroll\n      for(int j=0;j<9;++j) aa[j]=na[j];\n      #pragma unroll\n      for(int j=0;j<3;++j) bb[j]=nb[j];\n    }\n    __syncthreads();\n  }\n  if(t>=0) {\n    int gidx=(batch*T+t)*G+group;\n    int bidx=gidx*3;\n    int aidx=gidx*9;\n    int start=(batch*G+group)*3;\n    float prevc[3], prevn;\n    if(t==0) {\n      prevc[0]=c0[start];prevc[1]=c0[start+1];prevc[2]=c0[start+2];\n      prevn=1.f;\n    } else {\n      int previous=(batch*T+t-1)*G+group;\n      prevc[0]=cstate[previous*3];\n      prevc[1]=cstate[previous*3+1];\n      prevc[2]=cstate[previous*3+2];\n      prevn=nstate[previous];\n    }\n    #pragma unroll\n    for(int row=0;row<3;++row) {\n      gb[bidx+row]=bb[row];\n      #pragma unroll\n      for(int col=0;col<3;++col)\n        ga[aidx+row*3+col]=bb[row]*prevc[col];\n    }\n    gf[gidx]=ii*prevn;\n    gi[gidx]=ii;\n    if(t==0) {\n      #pragma unroll\n      for(int col=0;col<3;++col) {\n        gc0[start+col]=A[aidx+col]*bb[0]\n                       +A[aidx+3+col]*bb[1]\n                       +A[aidx+6+col]*bb[2];\n      }\n    }\n  }\n}\n'
# CuPy NVRTC compiles the CUDA code on the Colab GPU; it is not a simulated kernel.
(OUT/'pplstm_t4_fused_kernels.cu').write_text(CUDA_SOURCE)
try:
    KERNEL_FWD=cp.RawKernel(CUDA_SOURCE,'pplstm_fused_fwd',options=('--std=c++11',),backend='nvrtc')
    KERNEL_BWD=cp.RawKernel(CUDA_SOURCE,'pplstm_fused_bwd',options=('--std=c++11',),backend='nvrtc')
    # Force compile before CUDA Graph capture and outside the measured steps.
    KERNEL_FWD.compile()
    KERNEL_BWD.compile()
    KERNEL_READY=True
    print('Compiled both CUDA kernels successfully for',torch.cuda.get_device_name(0))
except Exception as exc:
    KERNEL_READY=False
    skip('compile_cuda_kernels',exc)

In [ ]:
def _cuda_arrays(*tensors):
    # torch.Tensor implements the CUDA DLPack protocol; no copies of GPU data.
    return [cp.from_dlpack(t.detach()) for t in tensors]

class FusedAffinePPLSTM(torch.autograd.Function):
    @staticmethod
    def forward(ctx,A,b,f,i,o,c0):
        assert KERNEL_READY, 'CUDA kernel did not compile'
        assert all(x.dtype==torch.float32 and x.is_cuda and x.is_contiguous() for x in (A,b,f,i,o,c0))
        B,T,G=A.shape[:3]
        assert 1<=T<=512,'T4 prototype currently supports length <=512'
        threads=1<<(T-1).bit_length()
        h=torch.empty((B,T,G*3),device=A.device,dtype=torch.float32)
        c=torch.empty_like(b)
        n=torch.empty_like(f)
        stream=cp.cuda.ExternalStream(torch.cuda.current_stream().cuda_stream)
        with stream:
            KERNEL_FWD((B*G,),(threads,),tuple(_cuda_arrays(A,b,f,i,o,c0,h,c,n))+ (np.int32(T),np.int32(G)),
                       shared_mem=threads*14*4,stream=stream)
        ctx.save_for_backward(A,f,o,c0,c,n)
        ctx.B,ctx.T,ctx.G=B,T,G
        return h
    @staticmethod
    def backward(ctx,dh):
        A,f,o,c0,c,n=ctx.saved_tensors
        B,T,G=ctx.B,ctx.T,ctx.G
        dh=dh.contiguous()
        ga=torch.empty_like(A);gb=torch.empty_like(c)
        gf=torch.empty_like(f);gi=torch.empty_like(f)
        go=torch.empty_like(o);gc0=torch.empty_like(c0)
        threads=1<<(T-1).bit_length()
        stream=cp.cuda.ExternalStream(torch.cuda.current_stream().cuda_stream)
        with stream:
            KERNEL_BWD((B*G,),(threads,),tuple(_cuda_arrays(A,f,o,c0,c,n,dh,ga,gb,gf,gi,go,gc0))+
                       (np.int32(T),np.int32(G)),shared_mem=threads*14*4,stream=stream)
        return ga,gb,gf,gi,go,gc0

class T4FusedPPLSTM(PPLSTM):
    # NO modifications to PPLSTM parameters, _token_maps, initialization, routing or gates.
    # Replaces only scan + initial memory application + normalizer + final output.
    def forward(self,x):
        A,b,f,i,o=self._token_maps(x)
        B,T,_=x.shape
        c0=self.initial_cell.tanh().reshape(1,self.groups,3).expand(B,-1,-1).contiguous()
        return FusedAffinePPLSTM.apply(A.contiguous(),b.contiguous(),f.contiguous(),
                                     i.contiguous(),o.contiguous(),c0)

def error_stats(ref,new):
    delta=(ref.detach().float()-new.detach().float())
    denom=ref.detach().float().norm().clamp_min(1e-8)
    return {'max_abs':float(delta.abs().max()),'relative_l2':float(delta.norm()/denom)}

def compare_reference_fused(width,length,batch=2):
    torch.manual_seed(14000+length)
    old=PPLSTM(width).to(DEVICE)
    new=T4FusedPPLSTM(width).to(DEVICE)
    new.load_state_dict(old.state_dict(),strict=True)
    inp=torch.randn(batch,length,width,device=DEVICE)
    x1=inp.detach().requires_grad_(True)
    x2=inp.detach().requires_grad_(True)
    weights=torch.randn_like(inp)
    y1=old(x1); y2=new(x2)
    l1=(y1*weights).mean()+y1.square().mean()*0.17
    l2=(y2*weights).mean()+y2.square().mean()*0.17
    l1.backward();l2.backward()
    torch.cuda.synchronize()
    out=error_stats(y1,y2)
    igrad=error_stats(x1.grad,x2.grad)
    byparam={name:error_stats(dict(old.named_parameters())[name].grad, p.grad)
             for name,p in new.named_parameters()}
    worst_abs=max(v['max_abs'] for v in byparam.values())
    worst_rel=max(v['relative_l2'] for v in byparam.values())
    # Conservative pilot tolerances. Report all comparisons; never hide near misses.
    ok=(out['max_abs']<5e-4 and out['relative_l2']<5e-3
        and igrad['max_abs']<1e-3 and igrad['relative_l2']<0.03
        and worst_abs<1e-3 and worst_rel<0.03)
    check(f'original_vs_fused_width{width}_length{length}',ok,output=out,
          input_gradient=igrad,worst_parameter_gradient_max_abs=worst_abs,
          worst_parameter_gradient_relative_l2=worst_rel,per_parameter=byparam)
    return ok

## 4 · Official xLSTM hybrid and the same four-block Transformer from the 200M experiment

In [ ]:
from xlstm import (xLSTMBlockStack, xLSTMBlockStackConfig, mLSTMBlockConfig,
                   mLSTMLayerConfig, sLSTMBlockConfig, sLSTMLayerConfig, FeedForwardConfig)

class PPLSTMReplacementLayer(nn.Module):
    """Adapt original sLSTM layer slot; retain convolution, group norm and dropout."""
    def __init__(self, original_layer, fused=False):
        super().__init__()
        self.config = original_layer.config
        self.conv1d = original_layer.conv1d if self.config.conv1d_kernel_size > 0 else None
        self.conv_act_fn = original_layer.conv_act_fn if self.conv1d is not None else None
        self.group_norm = original_layer.group_norm
        self.dropout = original_layer.dropout
        self.pplstm = (T4FusedPPLSTM if fused else PPLSTM)(self.config.embedding_dim)

    def forward(self, x, **kwargs):
        B,S,W=x.shape
        conv_x=self.conv_act_fn(self.conv1d(x)) if self.conv1d is not None else x
        y=self.dropout(self.pplstm(conv_x))
        H=self.config.num_heads
        y=y.reshape(B,S,H,W//H).transpose(1,2)
        y=self.group_norm(y).transpose(1,2).reshape(B,S,W)
        return y


def build_backbone(width=WIDTH, context=CONTEXT, fused=False):
    assert width%3==0 and width%HEADS==0
    cfg=xLSTMBlockStackConfig(
        mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig(
            conv1d_kernel_size=4, qkv_proj_blocksize=4, num_heads=HEADS)),
        slstm_block=sLSTMBlockConfig(slstm=sLSTMLayerConfig(
            backend='vanilla', num_heads=HEADS, conv1d_kernel_size=4,
            bias_init='powerlaw_blockdependent'),
            feedforward=FeedForwardConfig(proj_factor=1.3, act_fn='gelu')),
        context_length=context, num_blocks=DEPTH,
        embedding_dim=width, slstm_at=[1,3])
    stack=xLSTMBlockStack(cfg)
    assert list(stack.config.block_map)==[0,1,0,1]
    for index in (1,3):
        original=stack.blocks[index].xlstm
        if not hasattr(original,'slstm_cell'):
            raise RuntimeError('xLSTM API changed: expected slstm_cell in layer '+str(index))
        stack.blocks[index].xlstm=PPLSTMReplacementLayer(original, fused=fused)
    return stack

class PPLSTMLanguageModel(nn.Module):
    def __init__(self, vocab, width=WIDTH, context=CONTEXT, fused=False):
        super().__init__()
        self.vocab=vocab
        self.emb=nn.Embedding(vocab,width)
        nn.init.normal_(self.emb.weight,std=0.02)
        self.backbone=build_backbone(width,context,fused=fused)
        self.norm=nn.LayerNorm(width)
        self.out_bias=nn.Parameter(torch.zeros(vocab))
    def forward(self, tokens):
        h=self.norm(self.backbone(self.emb(tokens)))
        return F.linear(h,self.emb.weight,self.out_bias)  # tied LM output weights


# Attention uses native PyTorch SDPA, which dispatches supported CUDA kernels.
# No hand-written attention tensor BxHxTxT is constructed.
try:
    from torch.nn.attention import sdpa_kernel, SDPBackend
    HAVE_SDPA_CONTEXT=True
except ImportError:
    HAVE_SDPA_CONTEXT=False

SDPA_FUSED_AVAILABLE=False
SDPA_PROBE_NOTE='not tested'

def check_fused_sdpa(head_dim=16):
    """Confirm fused SDPA forward AND backward on the *actual* Kaggle GPU."""
    global SDPA_PROBE_NOTE
    if not HAVE_SDPA_CONTEXT:
        SDPA_PROBE_NOTE='sdpa_kernel context unavailable: automatic backend selection only'
        return False
    d=torch.float16 if USE_FP16_SDPA_ATTENTION else torch.float32
    q=torch.randn(2,4,128,head_dim,device=DEVICE,dtype=d,requires_grad=True)
    k=torch.randn_like(q,requires_grad=True);v=torch.randn_like(q,requires_grad=True)
    try:
        with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION,SDPBackend.EFFICIENT_ATTENTION]):
            y=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0)
            y.float().square().mean().backward()
        torch.cuda.synchronize()
        SDPA_PROBE_NOTE='fused SDPA forward+backward succeeded (exact Flash vs efficient choice is backend-selected)'
        return True
    except Exception as e:
        SDPA_PROBE_NOTE='fused SDPA not available for current input: '+str(e)[:250]
        return False

class FastCausalAttentionBlock(nn.Module):
    def __init__(self,width,num_heads,ffn_dim):
        super().__init__()
        assert width%num_heads==0
        self.nheads=num_heads;self.head_dim=width//num_heads
        self.ln1=nn.LayerNorm(width)
        self.qkv=nn.Linear(width,3*width)
        self.out=nn.Linear(width,width)
        self.ln2=nn.LayerNorm(width)
        self.ffn=nn.Sequential(nn.Linear(width,ffn_dim),nn.GELU(),nn.Linear(ffn_dim,width))
    def forward(self,x):
        b,t,c=x.shape
        q,k,v=self.qkv(self.ln1(x)).reshape(b,t,3,self.nheads,self.head_dim).unbind(2)
        q=q.transpose(1,2);k=k.transpose(1,2);v=v.transpose(1,2)
        if USE_FP16_SDPA_ATTENTION and x.device.type=='cuda':
            q,k,v=q.half(),k.half(),v.half()
        if SDPA_FUSED_AVAILABLE:
            with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION,SDPBackend.EFFICIENT_ATTENTION]):
                att=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=0.,is_causal=True)
        else:
            att=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=0.,is_causal=True)
        att=att.transpose(1,2).contiguous().view(b,t,c).to(x.dtype)
        x=x+self.out(att)
        return x+self.ffn(self.ln2(x))

class TransformerLM(nn.Module):
    def __init__(self,vocab,width,num_heads,ffn_dim,depth=DEPTH):
        super().__init__()
        self.vocab=vocab
        self.emb=nn.Embedding(vocab,width)
        nn.init.normal_(self.emb.weight,std=0.02)
        self.blocks=nn.ModuleList([FastCausalAttentionBlock(width,num_heads,ffn_dim) for _ in range(depth)])
        self.norm=nn.LayerNorm(width)
        self.out_bias=nn.Parameter(torch.zeros(vocab))
        # A position embedding is intentionally absent; use ALiBi/RoPE if desired in separate experiment.
        # Fixed/learned positional encodings are an important architectural confounder.
    def forward(self,tokens):
        x=self.emb(tokens)
        for block in self.blocks: x=block(x)
        x=self.norm(x)
        return F.linear(x,self.emb.weight,self.out_bias)

# IMPORTANT: For memory / order-sensitive language tasks, the transformer must have position information.
# We add causal ALiBi bias to SDPA as a deterministic, parameter-free relative-position scheme.
# The model source above will be adjusted before construction by applying the class below.
class FastCausalRotaryBlock(FastCausalAttentionBlock):
    def forward(self,x):
        b,t,c=x.shape
        q,k,v=self.qkv(self.ln1(x)).reshape(b,t,3,self.nheads,self.head_dim).unbind(2)
        q=q.transpose(1,2);k=k.transpose(1,2);v=v.transpose(1,2)
        if USE_FP16_SDPA_ATTENTION and x.device.type=='cuda':q,k,v=q.half(),k.half(),v.half()
        # Rotary embeddings provide relative position information and preserve fused causal SDPA support.
        q,k=apply_rotary(q,k)
        if SDPA_FUSED_AVAILABLE:
            with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION,SDPBackend.EFFICIENT_ATTENTION]):
                att=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=0.,is_causal=True)
        else:
            att=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=0.,is_causal=True)
        att=att.transpose(1,2).contiguous().view(b,t,c).to(x.dtype)
        x=x+self.out(att)
        return x+self.ffn(self.ln2(x))

def apply_rotary(q,k):
    # Standard pairwise rotary on head vectors, positions 0..T-1; no trainable parameters.
    d=q.shape[-1]
    if d%2:raise ValueError('Transformer head dim must be even for RoPE')
    pos=torch.arange(q.shape[-2],device=q.device,dtype=torch.float32)
    inv=1.0/(10000.0**(torch.arange(0,d,2,device=q.device,dtype=torch.float32)/d))
    ang=pos[:,None]*inv[None,:]
    cos=ang.cos().to(q.dtype)[None,None,:,:];sin=ang.sin().to(q.dtype)[None,None,:,:]
    def rot(x):
        even=x[...,0::2];odd=x[...,1::2]
        return torch.stack((even*cos-odd*sin,even*sin+odd*cos),dim=-1).flatten(-2)
    return rot(q),rot(k)

class RotaryTransformerLM(TransformerLM):
    def __init__(self,vocab,width,num_heads,ffn_dim,depth=DEPTH):
        super().__init__(vocab,width,num_heads,ffn_dim,depth)
        self.blocks=nn.ModuleList([FastCausalRotaryBlock(width,num_heads,ffn_dim) for _ in range(depth)])

def transformer_match(vocab,target,probe_classes=None):
    """Exactly count parameters; prefer head_dim=16 fused-friendly width when within 0.75%."""
    best=None
    for w in (64,80,72,96):
        for nhead in (4,):
            if w%nhead:continue
            for ffd in range(96,768,8):
                # Each residual block: two LN, QKV, output projection and 2-layer MLP.
                block=4*w*w+2*w*ffd+9*w+ffd
                head=(vocab if probe_classes is None else probe_classes*w+probe_classes)
                n=vocab*w+2*w+DEPTH*block+head
                gap=abs(n-target)
                # Matching to within 0.75% is sufficient; then favor width64/head_dim16.
                within=gap/target<=0.0075
                key=(0 if within and w==64 else 1 if within else 2,gap,abs(ffd-256))
                if best is None or key<best[0]:best=(key,w,nhead,ffd,n)
    _,w,nhead,ffd,n=best
    return {'width':w,'heads':nhead,'ffn':ffd,'params':n,'target':target,
            'param_difference_pct':100*(n-target)/target}

## 5 · FineWeb-Edu input preparation (same document hash split and tokenizer format as the 200M notebook)

In [ ]:
"""Frozen FineWeb-Edu sample-10BT data pipeline.
Only the dataset/test *types* were taken from the provided example notebook.
No model code or experiment results from that notebook are imported.
"""
import glob,hashlib,json,os,time
from pathlib import Path
import numpy as np


def sha_text(x):
    return hashlib.sha256(x.encode('utf8','ignore')).hexdigest()


def split_doc(text):
    """Exact duplicates land in the same split; near-duplicates remain a caveat."""
    r=int(sha_text(text)[:12],16)%1000
    return 'valid' if r<10 else 'test' if r<20 else 'train'


def documents(paths):
    import pyarrow.parquet as pq
    for filename in paths:
        pqfile=pq.ParquetFile(str(filename))
        if 'text' not in pqfile.schema.names:
            raise ValueError(f'Not a FineWeb text Parquet shard: {filename}')
        for batch in pqfile.iter_batches(columns=['text'],batch_size=512):
            for raw in batch.column(0).to_pylist():
                if isinstance(raw,str) and len(raw.strip())>=20:
                    yield raw.replace('\x00',' ').strip()


def find_paths(pattern):
    found=[Path(p) for p in sorted(glob.glob(pattern,recursive=True)) if Path(p).is_file()]
    if not found:
        raise FileNotFoundError('FineWeb-Edu parquet not found. Attach sample/10BT/000_00000.parquet in Kaggle Inputs, then edit PARQUET_GLOB. Current: '+pattern)
    if len(found)>1:
        filtered=[p for p in found if 'fineweb' in str(p).lower()]
        if filtered:found=filtered
        else:raise RuntimeError('Multiple generic parquet shards matched. Narrow PARQUET_GLOB to FineWeb-Edu sample-10BT only.')
    return found


def manifest_for_input(paths):
    rows=[]
    for p in paths:
        with p.open('rb') as stream:head=stream.read(1024*1024)
        rows.append({'path':str(p),'file_size':p.stat().st_size,
             'first_megabyte_sha256':hashlib.sha256(head).hexdigest()})
    return rows


def prepare(root,glob_pattern,existing_data_root='',vocab=4096,max_train=30_000_000,
            valid_count=250_000,test_count=250_000,sample_chars=3_000_000,
            stop_epoch=None):
    import sentencepiece as spm
    out=Path(root)/'data';out.mkdir(parents=True,exist_ok=True)
    if existing_data_root:
        base=Path(existing_data_root)/'data'
        if not (base/'spm.model').exists() or not (base/'train.bin').exists():
            raise ValueError('EXISTING_DATA_ROOT must contain data/spm.model and train/valid/test.bin')
        manifest=json.loads((base/'manifest.json').read_text()) if (base/'manifest.json').exists() else {}
        return {'root':str(base),'tokenizer':str(base/'spm.model'),'sizes':{
            part:(base/f'{part}.bin').stat().st_size//2 for part in ('train','valid','test')},
            'reused':True,'original_manifest':manifest,'source':'FineWeb-Edu, externally prepared corpus (verify uploaded provenance)'}

    old=out/'manifest.json'
    if old.exists() and (out/'spm.model').exists():
        manifest=json.loads(old.read_text())
        if manifest.get('complete') and all((out/f'{k}.bin').exists() for k in ('train','valid','test')):
            return manifest
    paths=find_paths(glob_pattern)
    source_manifest=manifest_for_input(paths)
    sample=out/'spm_training_sample.txt'
    if not (out/'spm.model').exists():
        chars=0;count=0
        with sample.open('w',encoding='utf8') as f:
            for t in documents(paths):
                if split_doc(t)!='train':continue
                t=' '.join(t[:15000].split());f.write(t+'\n');chars+=len(t);count+=1
                if chars>=sample_chars:break
        if chars<20000:raise RuntimeError('Not enough training text to fit a tokenizer')
        spm.SentencePieceTrainer.Train(input=str(sample),model_prefix=str(out/'spm'),
            vocab_size=vocab,model_type='bpe',character_coverage=0.9995,
            unk_id=0,eos_id=1,bos_id=-1,pad_id=-1,
            max_sentence_length=16384,hard_vocab_limit=False, num_threads=4)
        sample.unlink(missing_ok=True)
    sp=spm.SentencePieceProcessor(model_file=str(out/'spm.model'))
    if sp.vocab_size()>65535:raise ValueError('uint16 token array requires vocab<=65535')
    sizes={'train':int(max_train),'valid':int(valid_count),'test':int(test_count)}
    mm={}
    for k,n in sizes.items():
        mm[k]=np.memmap(out/f'{k}.bin',dtype=np.uint16,mode='w+',shape=(n,))
    pos={k:0 for k in sizes}
    doc_count={k:0 for k in sizes}
    print('Tokenizing attached FineWeb-Edu. Caps:',sizes,flush=True)
    for j,t in enumerate(documents(paths)):
        if stop_epoch is not None and time.time()>stop_epoch:
            print('Corpus preparation budget reached; using filled prefixes, if sufficient.',flush=True)
            break
        split=split_doc(t)
        remain=sizes[split]-pos[split]
        if remain<=0:
            if all(pos[k]>=sizes[k] for k in sizes):break
            continue
        ids=sp.encode(t,out_type=int)
        if not ids:continue
        ids.append(sp.eos_id())
        take=min(len(ids),remain)
        mm[split][pos[split]:pos[split]+take]=np.asarray(ids[:take],dtype=np.uint16)
        pos[split]+=take;doc_count[split]+=1
        if j and j%5000==0:print('data',j,'documents',pos,flush=True)
    for k in sizes:
        mm[k].flush();del mm[k]
        with (out/f'{k}.bin').open('r+b') as f:f.truncate(pos[k]*2)
    if pos['train']<512*128 or min(pos['valid'],pos['test'])<4096:
        raise RuntimeError('Not enough authentic FineWeb-Edu text before preparation deadline: '+str(pos))
    manifest={'complete':True,'root':str(out),'tokenizer':str(out/'spm.model'),
        'dataset':'HuggingFaceFW/fineweb-edu sample-10BT','data_provenance':source_manifest,
        'document_hash_split':'1% validation, 1% test, 98% train; exact hash by full text',
        'document_counts':doc_count,'sizes':pos,'vocab_size':sp.vocab_size(),
        'reused':False,'tokens_are_bpe':True,
        'limits':sizes,'sha256_tokenizer':hashlib.sha256((out/'spm.model').read_bytes()).hexdigest()}
    old.write_text(json.dumps(manifest,indent=2))
    return manifest


def load_tokens(manifest,part):
    f=Path(manifest['root'])/f'{part}.bin'
    return np.memmap(f,mode='r',dtype=np.uint16)

In [ ]:
# Never fall back to Tiny Shakespeare silently.
def locate_prepared():
    bases=[]
    if PREPARED_DIR:bases.append(Path(PREPARED_DIR))
    for path in glob.glob('/kaggle/input/**/train.bin',recursive=True):bases.append(Path(path).parent)
    bases.append(OUT/'data')
    for folder in dict.fromkeys(bases):
        if all((folder/p).exists() for p in ('train.bin','valid.bin','test.bin','spm.model')):
            return folder
    return None

prepared=locate_prepared()
if prepared is not None:
    import sentencepiece as spm
    processor=spm.SentencePieceProcessor(model_file=str(prepared/'spm.model'))
    VOCAB=processor.vocab_size()
    TOKENS={p:np.memmap(prepared/f'{p}.bin',mode='r',dtype=np.uint16) for p in ('train','valid','test')}
    DATA_INFO={'mode':'reused_prepared','prepared':str(prepared),'sizes':{k:len(x) for k,x in TOKENS.items()},
               'tokenizer_sha256':hashlib.sha256((prepared/'spm.model').read_bytes()).hexdigest()}
else:
    parquet_paths=glob.glob(TRAIN_PARQUET_GLOB,recursive=True)
    if len(parquet_paths)!=1:
        raise FileNotFoundError(f'Attach Kaggle dataset cartman with one 000_00000.parquet; matched {len(parquet_paths)}: {parquet_paths[:6]}')
    manifest=prepare(str(OUT),TRAIN_PARQUET_GLOB,vocab=4096,max_train=MAX_PREP_TRAIN_TOKENS,
                     valid_count=150_000,test_count=150_000)
    TOKENS={p:load_tokens(manifest,p) for p in ('train','valid','test')}
    VOCAB=int(manifest['vocab_size'])
    DATA_INFO={'mode':'attached_fineweb_parquet','manifest':manifest,'sizes':{k:len(x) for k,x in TOKENS.items()}}
assert min(map(len,TOKENS.values()))>=CONTEXT+10000,'Not enough held-out tokens'
assert max(int(np.max(v)) for v in TOKENS.values()) < VOCAB,'Tokenizer/data mismatch'
DATA_INFO['fingerprint']=hashlib.sha256(
    np.asarray(TOKENS['train'][:524288],dtype=np.uint16).tobytes()
    +str(VOCAB).encode()).hexdigest()
save_json('dataset_info.json',DATA_INFO)
print('FineWeb-Edu dataset:',DATA_INFO['mode'],'vocab',VOCAB,'tokens',DATA_INFO['sizes'])
print('10M token presentations PER MODEL; distinct prepared train tokens:',len(TOKENS['train']))

## 6 · Compile validation and strict same-weight forward/backward audits

In [ ]:
assert KERNEL_READY, 'Fused T4 CUDA kernel failed to compile; cannot make a valid comparison.'
# Direct PPLSTM cell tests including all gate / input gradients.
original_cpu=PPLSTM(12).to('cpu',dtype=torch.float64)
x_cpu=torch.randn(2,17,12,dtype=torch.float64,requires_grad=True)
ya=original_cpu(x_cpu); yb=original_cpu.serial_reference(x_cpu)
diff=float((ya-yb).abs().max())
ga=torch.autograd.grad(ya.square().mean(),(x_cpu,*original_cpu.parameters()),retain_graph=True)
gb=torch.autograd.grad(yb.square().mean(),(x_cpu,*original_cpu.parameters()))
gdiff=max(float((a-b).abs().max()) for a,b in zip(ga,gb))
assert check('published_scan_vs_serial_fp64',diff<1e-9 and gdiff<1e-9,max_abs_output=diff,max_abs_gradient=gdiff)
for length in (8,128,512):
    assert compare_reference_fused(WIDTH,length,batch=2),f'Direct PPLSTM GPU verification failed for length {length}'
# Same-weight audits below use separately instantiated shadow models, never the training Graph models.
SDPA_FUSED_AVAILABLE=check_fused_sdpa(head_dim=16)
print('Transformer SDPA:',SDPA_PROBE_NOTE)
HYBRID_ORIGINAL=PPLSTMLanguageModel(VOCAB,fused=False).to(DEVICE)
HYBRID_FUSED=PPLSTMLanguageModel(VOCAB,fused=True).to(DEVICE)
HYBRID_FUSED.load_state_dict(HYBRID_ORIGINAL.state_dict(),strict=True)
MATCH=transformer_match(VOCAB,nparams(HYBRID_ORIGINAL))
TRANSFORMER=RotaryTransformerLM(VOCAB,MATCH['width'],MATCH['heads'],MATCH['ffn']).to(DEVICE)
print('PARAMETERS original / fused / Transformer:',nparams(HYBRID_ORIGINAL),nparams(HYBRID_FUSED),nparams(TRANSFORMER),'match details:',MATCH)
assert nparams(HYBRID_ORIGINAL)==nparams(HYBRID_FUSED)
assert abs(MATCH['param_difference_pct'])<1.5, 'Transformer parameter count differs >1.5%'
assert all(torch.equal(a,b) for a,b in zip(HYBRID_ORIGINAL.state_dict().values(),HYBRID_FUSED.state_dict().values()))
AUDIT_ORIGINAL=PPLSTMLanguageModel(VOCAB,fused=False).to(DEVICE)
AUDIT_FUSED=PPLSTMLanguageModel(VOCAB,fused=True).to(DEVICE)
AUDIT_ROWS=[]

def stat_error(ref,new):
    delta=ref.detach().float()-new.detach().float()
    return {'max_abs':float(delta.abs().max()),'relative_l2':float(delta.norm()/ref.detach().float().norm().clamp_min(1e-8))}

def audit_same_weights(state,tag,example_x,example_y,fail_on_error=True):
    # IMPORTANT: two scratch/shadow models get SAME state. Does not alter either training runner.
    AUDIT_ORIGINAL.load_state_dict(state,strict=True)
    AUDIT_FUSED.load_state_dict(state,strict=True)
    AUDIT_ORIGINAL.train();AUDIT_FUSED.train()
    AUDIT_ORIGINAL.zero_grad(set_to_none=True);AUDIT_FUSED.zero_grad(set_to_none=True)
    first=example_x.clone().detach();second=example_x.clone().detach()
    out_a=AUDIT_ORIGINAL(first);out_b=AUDIT_FUSED(second)
    loss_a=F.cross_entropy(out_a.reshape(-1,VOCAB),example_y.reshape(-1),ignore_index=-100)
    loss_b=F.cross_entropy(out_b.reshape(-1,VOCAB),example_y.reshape(-1),ignore_index=-100)
    loss_a.backward();loss_b.backward()
    torch.cuda.synchronize()
    out_err=stat_error(out_a,out_b)
    pa=dict(AUDIT_ORIGINAL.named_parameters());pb=dict(AUDIT_FUSED.named_parameters())
    assert list(pa)==list(pb)
    total_num=0.;total_den=0.;worst_abs=0.;worst_rel=0.;worst_name=''
    param_errors=[]
    for name in pa:
        a=pa[name].grad;b=pb[name].grad
        assert a is not None and b is not None, 'No gradient: '+name
        diff=(a.float()-b.float());total_num+=float(diff.square().sum());total_den+=float(a.float().square().sum())
        abs_err=float(diff.abs().max());rel_err=float(diff.norm()/a.float().norm().clamp_min(1e-8))
        param_errors.append({'audit':tag,'parameter':name,'max_abs':abs_err,'rel_l2':rel_err})
        if abs_err>worst_abs:worst_abs=abs_err;worst_name=name
        worst_rel=max(worst_rel,rel_err)
    rel_all=math.sqrt(total_num/max(total_den,1e-20))
    # A practical FP32 tolerance, explicitly logged. Bitwise exactness is not expected.
    passed=(out_err['max_abs']<2e-3 and out_err['relative_l2']<3e-4
            and worst_abs<2e-3 and rel_all<3e-4 and math.isfinite(rel_all))
    result={'audit':tag,'passed':bool(passed),'source_weights_equal':True,
            'logit_max_abs':out_err['max_abs'],'logit_rel_l2':out_err['relative_l2'],
            'loss_original':float(loss_a),'loss_fused':float(loss_b),'loss_abs_diff':abs(float(loss_a)-float(loss_b)),
            'grad_rel_l2_all':rel_all,'grad_worst_abs':worst_abs,'grad_worst_relative_per_parameter':worst_rel,
            'grad_worst_absolute_parameter':worst_name}
    AUDIT_ROWS.append(result)
    save_csv('trained_weight_same_state_audits.csv',AUDIT_ROWS)
    if fail_on_error and not passed:
        check(tag+'_same_weights',False,**{k:v for k,v in result.items() if k!='passed'})
        raise RuntimeError('Numerical kernel audit failed: '+tag+'. Do not trust this fused implementation for training.')
    check(tag+'_same_weights',passed,**{k:v for k,v in result.items() if k!='passed'})
    return result

# Run before training and repeat periodically using the ORIGINAL and FUSED trained weights.
probe_x=torch.randint(0,VOCAB,(2,CONTEXT),device=DEVICE)
probe_y=torch.randint(0,VOCAB,(2,CONTEXT),device=DEVICE)
audit_same_weights(HYBRID_ORIGINAL.state_dict(),'initial_original_weights',probe_x,probe_y)
audit_same_weights(HYBRID_FUSED.state_dict(),'initial_fused_weights',probe_x,probe_y)
save_json('parameter_matching.json',{'hybrid':nparams(HYBRID_ORIGINAL),'fused':nparams(HYBRID_FUSED),'transformer':nparams(TRANSFORMER),**MATCH})

## 7 · CUDA Graphs, optimizer, identical example order and held-out evaluation

In [ ]:
class FwdBwdGraph:
    def __init__(self,model,example_x,example_y):
        self.model=model
        self.static_x=torch.zeros_like(example_x)
        self.static_y=torch.zeros_like(example_y)
        self.graph=torch.cuda.CUDAGraph()
        warm_stream=torch.cuda.Stream(device=DEVICE)
        warm_stream.wait_stream(torch.cuda.current_stream())
        model.train()
        with torch.cuda.stream(warm_stream):
            for _ in range(3):
                model.zero_grad(set_to_none=True)
                self.loss_fn().backward()
        torch.cuda.current_stream().wait_stream(warm_stream)
        model.zero_grad(set_to_none=True)
        with torch.cuda.graph(self.graph,stream=warm_stream):
            self.loss=self.loss_fn()
            self.loss.backward()
    def loss_fn(self):
        logits=self.model(self.static_x)
        return F.cross_entropy(logits.reshape(-1,logits.shape[-1]),self.static_y.reshape(-1),ignore_index=-100)
    def replay(self,x,y):
        assert x.shape==self.static_x.shape and y.shape==self.static_y.shape
        self.static_x.copy_(x);self.static_y.copy_(y)
        self.graph.replay()
        return self.loss

def lm_batch(step,count=None):
    rng=np.random.default_rng(SEED+98765431*int(step))
    arr=TOKENS['train']
    max_start=(len(arr)-CONTEXT-1)//CONTEXT
    ids=rng.integers(0,max_start,size=BATCH)
    w=np.stack([np.asarray(arr[int(j)*CONTEXT:int(j)*CONTEXT+CONTEXT+1],dtype=np.int64) for j in ids])
    x=torch.from_numpy(w[:,:-1].copy()).to(DEVICE)
    y=torch.from_numpy(w[:,1:].copy()).to(DEVICE)
    if count is not None and count<BATCH*CONTEXT:y.reshape(-1)[count:]=-100
    return x,y

def make_optimizer(model):
    try:return torch.optim.AdamW(model.parameters(),lr=8e-4,betas=(0.9,0.95),weight_decay=.01,fused=True)
    except (TypeError,RuntimeError):return torch.optim.AdamW(model.parameters(),lr=8e-4,betas=(0.9,0.95),weight_decay=.01)

def learning_rate(tokens):
    warm=500_000;maximum=8e-4;minimum=8e-5
    if tokens<warm:return maximum*max(1,tokens+CONTEXT)/warm
    prog=max(0.,min(1.,(tokens-warm)/max(1,TARGET_TOKENS_PER_MODEL-warm)))
    return minimum+.5*(maximum-minimum)*(1.+math.cos(math.pi*prog))

def step_update(r,x,y):
    model=r['model'];opt=r['optimizer']
    if r['graph'] is not None:
        loss=r['graph'].replay(x,y)
    else:
        opt.zero_grad(set_to_none=True)
        scores=model(x)
        loss=F.cross_entropy(scores.reshape(-1,VOCAB),y.reshape(-1),ignore_index=-100)
        loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    opt.step()
    return loss

@torch.no_grad()
def evaluate(model,split='valid',limit=SMALL_VALIDATION_TOKENS):
    model.eval();arr=TOKENS[split]
    windows=min(limit//CONTEXT,(len(arr)-1)//CONTEXT)
    assert windows>0
    total_loss=0.;correct=0;count=0
    for start in range(0,windows,BATCH):
        size=min(BATCH,windows-start)
        windows_np=np.stack([np.asarray(arr[(start+j)*CONTEXT:(start+j)*CONTEXT+CONTEXT+1],dtype=np.int64) for j in range(size)])
        x=torch.from_numpy(windows_np[:,:-1].copy()).to(DEVICE)
        y=torch.from_numpy(windows_np[:,1:].copy()).to(DEVICE)
        logits=model(x)
        total_loss+=float(F.cross_entropy(logits.reshape(-1,VOCAB),y.reshape(-1),reduction='sum').item())
        correct+=int((logits.argmax(-1)==y).sum().item());count+=y.numel()
    model.train()
    value=total_loss/count
    return {'n_tokens':count,'nll':value,'perplexity':math.exp(min(30,value)),'top1_pct':100*correct/count}

RUNNERS={}
models=[('hybrid_original_graph',HYBRID_ORIGINAL),('hybrid_fused_graph',HYBRID_FUSED),('transformer_sdpa',TRANSFORMER)]
for name,model in models:
    optim=make_optimizer(model)
    ex_x,ex_y=lm_batch(0)
    try:
        graph=FwdBwdGraph(model,ex_x,ex_y)
        engine='cuda_graph_forward_backward'
    except Exception as exc:
        if name.startswith('hybrid') and REQUIRE_CUDA_GRAPHS_FOR_HYBRIDS:
            raise RuntimeError('CUDA Graph capture failed for '+name+': '+repr(exc)) from exc
        graph=None;engine='eager_fallback'
        print('Graph unavailable',name,repr(exc))
    RUNNERS[name]={'model':model,'optimizer':optim,'graph':graph,'engine':engine,'seen':0,'steps':0,'train_seconds':0.0,'last_loss':None}
    check(name+'_engine',True,engine=engine)

# Before training: run both graphs on the EXACT same nontraining batch WITHOUT optimizer steps.
a,b=lm_batch(4)
la=float(RUNNERS['hybrid_original_graph']['graph'].replay(a,b).item())
lb=float(RUNNERS['hybrid_fused_graph']['graph'].replay(a,b).item())
check('captured_graph_loss_agreement',abs(la-lb)<2e-3,loss_original=la,loss_fused=lb,absolute_difference=abs(la-lb))
assert not CHECK_FAILURES,'Correctness failed before training: '+str(CHECK_FAILURES)
TRAIN_CURVE=[];DRIFT=[]
def weight_drift():
    a=dict(HYBRID_ORIGINAL.named_parameters());b=dict(HYBRID_FUSED.named_parameters())
    assert list(a)==list(b)
    num=den=0.;largest=0.;worst=''
    with torch.no_grad():
        for k in a:
            diff=(a[k].float()-b[k].float())
            num+=float(diff.square().sum());den+=float(a[k].float().square().sum())
            ma=float(diff.abs().max())
            if ma>largest:largest=ma;worst=k
    return {'weight_rel_l2':math.sqrt(num/max(den,1e-20)),'weight_max_abs':largest,'weight_worst_param':worst}

save_json('experiment_config.json',{'token_budget_per_model':TARGET_TOKENS_PER_MODEL,'batch':BATCH,'context':CONTEXT,
   'depth':DEPTH,'heads':HEADS,'hybrid_width':WIDTH,'seed':SEED,'training_optimizer':'AdamW lr 8e-4 warmup 500k cosine, beta (.9,.95), weight decay 0.01, clip 1',
   'kernel':'Experimental CuPy RawKernel CUDA T4, FP32 forward+backward; PPLSTM scan/normalizer/output only',
   'original_pplstm':'Unmodified PPLSTM _token_maps, gates, permutations, initialization',
   'transformer_attention':'RoPE + FP16 SDPA when GPU supports it',
   'sdpa_backend':SDPA_PROBE_NOTE,'data_info':DATA_INFO,'parameter_match':MATCH,
   'limits':['Training drift may accumulate from harmless floating-point order differences',
       'Same-weight audits, NOT merely comparing independently trained weights, establish local equivalence',
       'Optimizers outside CUDA Graph capture; fused kernel still experimental',
       'Single seed, one data sample, nonmatched architecture except parameter count']})
bundle()

## 8 · Train all three on identical 10M-token presentation streams; re-check fusion using trained weights

In [ ]:
steps_target=math.ceil(TARGET_TOKENS_PER_MODEL/(BATCH*CONTEXT))
if RUN_SMOKE_ONLY:steps_target=min(SMOKE_STEPS,steps_target)
print('Planned',steps_target,'steps per model; nominal target',format(TARGET_TOKENS_PER_MODEL,','),'tokens/model. Smoke only:',RUN_SMOKE_ONLY)
next_audit=AUDIT_EVERY_TOKENS
completed_steps=0
for start in range(0,steps_target,REPORT_EVERY_STEPS):
    if time.monotonic()>=DEADLINE-120:
        print('TIME GUARD: stopping at a completed shared training chunk. Final partial reports will be exported.')
        break
    end=min(steps_target,start+REPORT_EVERY_STEPS)
    # Alternate model order by chunk to reduce consistent thermal / order bias.
    order=list(RUNNERS)
    if (start//REPORT_EVERY_STEPS)%2==1:order.reverse()
    for name in order:
        r=RUNNERS[name]
        torch.cuda.synchronize()
        t0=time.perf_counter()
        for step in range(start,end):
            remaining=TARGET_TOKENS_PER_MODEL-step*BATCH*CONTEXT
            if remaining<=0:break
            count=min(BATCH*CONTEXT,remaining)
            x,y=lm_batch(step,count)
            for group in r['optimizer'].param_groups:group['lr']=learning_rate(step*BATCH*CONTEXT)
            loss=step_update(r,x,y)
            r['steps']+=1;r['seen']+=count
        torch.cuda.synchronize()
        r['train_seconds']+=time.perf_counter()-t0
        r['last_loss']=float(loss.item())
        assert math.isfinite(r['last_loss']),'Nonfinite training loss: '+name
    completed_steps=end
    tokens=min(TARGET_TOKENS_PER_MODEL,end*BATCH*CONTEXT)
    for name,r in RUNNERS.items():
        met=evaluate(r['model'],split='valid',limit=SMALL_VALIDATION_TOKENS)
        line={'model':name,'tokens_presented':r['seen'],'training_steps':r['steps'],'engine':r['engine'],
              'last_train_loss':r['last_loss'],'step_train_s':r['train_seconds'],
              'step_train_tok_per_s':r['seen']/max(1e-9,r['train_seconds']),**met}
        TRAIN_CURVE.append(line)
        print(f"{tokens:,} {name}: val PPL {met['perplexity']:.2f}; acc {met['top1_pct']:.2f}%; train tok/s {line['step_train_tok_per_s']:.0f}",flush=True)
    drift={'tokens':tokens,**weight_drift()}
    DRIFT.append(drift)
    print('Original vs fused independent-training weight drift:',drift,flush=True)
    save_csv('training_validation_curve.csv',TRAIN_CURVE)
    save_csv('trained_weight_drift.csv',DRIFT)
    if tokens>=next_audit or end==steps_target:
        # Two audits: original-trained weights and fused-trained weights, each in BOTH implementations.
        xa,ya=lm_batch(10000000+end)
        audit_same_weights(HYBRID_ORIGINAL.state_dict(),f'original_trained_{tokens}',xa[:2].contiguous(),ya[:2].contiguous())
        audit_same_weights(HYBRID_FUSED.state_dict(),f'fused_trained_{tokens}',xa[:2].contiguous(),ya[:2].contiguous())
        next_audit=((tokens//AUDIT_EVERY_TOKENS)+1)*AUDIT_EVERY_TOKENS
    bundle()
    if CHECK_FAILURES:
        raise RuntimeError('Kernel correctness check failed; stopped: '+str(CHECK_FAILURES))

# Final summary, including partial completion if time guard triggered.
FINAL=[]
for name,r in RUNNERS.items():
    valid=evaluate(r['model'],'valid',FINAL_EVAL_TOKENS)
    test=evaluate(r['model'],'test',FINAL_EVAL_TOKENS)
    finish={'model':name,'status':'complete' if r['seen']==TARGET_TOKENS_PER_MODEL else 'partial_or_smoke',
            'tokens_presented':r['seen'],'optimizer_steps':r['steps'],'parameters':nparams(r['model']),
            'engine':r['engine'],'training_wall_seconds_excluding_validation':r['train_seconds'],
            'training_tokens_per_s':r['seen']/max(r['train_seconds'],1e-9),
            **{'valid_'+key:value for key,value in valid.items()},
            **{'test_'+key:value for key,value in test.items()}}
    FINAL.append(finish)
    if SAVE_FINAL_CHECKPOINTS:
        torch.save({'model':r['model'].state_dict(),'optimizer':r['optimizer'].state_dict(),'metadata':finish,'data_fingerprint':DATA_INFO['fingerprint']},OUT/(name+'_checkpoint.pt'))
    print('FINAL',finish,flush=True)
save_csv('final_language_results.csv',FINAL)
save_json('summary.json',{'models':FINAL,'audit_results':AUDIT_ROWS,
       'independent_weight_drift':DRIFT[-1] if DRIFT else None,
       'all_numerical_tests_passed':all(item['passed'] for item in CHECKS),
       'complete':all(f['status']=='complete' for f in FINAL),'smoke_only':RUN_SMOKE_ONLY,
       'total_wall_hours':(time.monotonic()-RUN_START)/3600,
       'notes':['Kernel fusion keeps PPLSTM token maps and parameters unchanged',
                'A successfully captured CUDA Graph is distinct from a custom CUDA kernel',
                'GPU benchmark and full training speeds include gradient clip + AdamW, exclude validation/data preparation',
                'Validated same-weight outputs/gradients do not guarantee bit-for-bit same trajectories']})
print('RESULT ZIP:',bundle(),'size bytes:',(OUT/'PPLSTM_Transformer_Fusion_10M_Results.zip').stat().st_size,flush=True)

## 9 · How to interpret the ZIP

The small `PPLSTM_Transformer_Fusion_10M_Results.zip` will be in Kaggle **Output → `pplstm_fusion_10m/`**. Download it and upload here.

- `trained_weight_same_state_audits.csv` is the **key maths check**. Each row compares **identical trained weights** using original vs fused kernels. Inspect `logit_max_abs`, `grad_rel_l2_all`, `loss_abs_diff`, and `passed`.
- `trained_weight_drift.csv` measures how far the independently trained original/fused models diverged. Some divergence is expected from floating-point operation order; it is **not itself evidence of a mathematical change**.
- `training_validation_curve.csv` and `final_language_results.csv` show held-out NLL/perplexity/accuracy and tokens/sec for all three models.
- `correctness_checks.json` documents initial direct cell checks (8/128/512), serial/parallel checks and trained-state audits.
- `experiment_config.json` records dataset fingerprint, model sizes and compiler/GPU details. `pplstm_t4_fused_kernels.cu` includes the actual experimental CUDA source.
- `.pt` model checkpoints are **not** added to the ZIP; they stay in Kaggle Output separately.

**Fairness limits:** Both hybrids share architecture, initialization, data and optimizer schedule; they differ only in PPLSTM memory scan implementation. The Transformer is a parameter-matched *different architecture* using RoPE and potentially FP16 attention. Its accuracy and speed are useful context but cannot establish whether PPLSTM fusion changed its mathematics. This experiment tests one seed, short 128-token contexts and T4 FP32; it does not guarantee stability at all parameter values, lengths, or GPU types.